<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/direction_tuning_three_modalities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Direction tuning across three recording scales

A confidence-building notebook for the [OpenScope Community Predictive
Processing](https://allenneuraldynamics.github.io/openscope-community-predictive-processing/)
dataset (companion to the receptive-field notebook). Direction/orientation
selectivity is the canonical property of mouse visual cortex, so it is an ideal
known-answer check.

### Direction vs. orientation — an important distinction

The 14-condition grating sweep in this dataset is presented as **drifting
gratings** (temporal frequency = 2 Hz) in all three modalities. Drifting gratings
natively probe **direction tuning** (0–360°): a neuron can distinguish a grating
moving up-left from the same grating moving down-right. *Orientation* tuning proper
(indifferent to drift direction, 0–180°) is best measured with **static** gratings,
which are **not** available as a sweep here (the static/TF=0 trials in these blocks
are all a single orientation). We therefore:

- report **DSI (direction selectivity index)** as the primary metric,
- report **OSI** only as a value **derived** by folding the 360° tuning curve to
  180° (the standard 2θ vector method) — not as a static-grating measurement.

| Modality | DANDI | Signal |
|---|---|---|
| Neuropixels ecephys | [001637](https://dandiarchive.org/dandiset/001637) | spike rate |
| Mesoscope 2-photon | [001768](https://dandiarchive.org/dandiset/001768) | ΔF/F (jGCaMP8s, soma) |
| SLAP2 | [001424](https://dandiarchive.org/dandiset/001424) | ΔF/F (iGluSnFR, glutamate) |

**Avoiding over-selection.** Ranking example cells by a selectivity index favours
spiky, one-lobe curves that plot as near-lines. Instead we **fit a double von Mises
model** to every tuning curve, **select examples by fit quality (R²)**, and report
the fitted **tuning half-width (HWHM)** so a genuinely narrow curve is visible as
narrow rather than mistaken for the whole population.

**Runtime:** CPU is fine; ~8–12 min end to end.

In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy

In [ ]:
#@title Streaming helpers + von Mises tuning model
import h5py, remfile, requests, numpy as np, pandas as pd
from scipy.optimize import curve_fit

def s3_url(dandiset, asset_id, version="draft"):
    r = requests.get(f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/",
                     allow_redirects=False, timeout=30)
    return r.headers["Location"]
def open_nwb(dandiset, asset_id):
    return h5py.File(remfile.File(s3_url(dandiset, asset_id)), "r")
def col(group, name):
    v = group[name][:]
    return np.array([x.decode() if isinstance(x, bytes) else x for x in v])

def resp_trace(trace, ts, times, resp, base):
    """Baseline-subtracted mean response per trial (timestamps assumed sorted)."""
    lo_r=np.searchsorted(ts,times+resp[0]); hi_r=np.searchsorted(ts,times+resp[1])
    lo_b=np.searchsorted(ts,times+base[0]); hi_b=np.searchsorted(ts,times+base[1])
    out=np.full(len(times),np.nan)
    for k in range(len(times)):
        if hi_r[k]>lo_r[k] and hi_b[k]>lo_b[k]:
            out[k]=np.nanmean(trace[lo_r[k]:hi_r[k]])-np.nanmean(trace[lo_b[k]:hi_b[k]])
    return out

def dvm(theta, a1, a2, kappa, theta_p, base):
    """Double von Mises: two lobes 180 deg apart, shared width (Mazurek/Carandini)."""
    return (a1*np.exp(kappa*(np.cos(theta-theta_p)-1))
            + a2*np.exp(kappa*(np.cos(theta-theta_p-np.pi)-1)) + base)
def fit_dvm(curve, theta):
    A0=max(np.clip(curve,0,None).max(),1e-3); tp0=theta[np.argmax(np.clip(curve,0,None))]
    try:
        popt,_=curve_fit(dvm, theta, curve, p0=[A0,A0*0.4,2.0,tp0,max(np.median(curve),0)],
            bounds=([0,0,0.3,-np.pi,-abs(curve).max()],[np.inf,np.inf,50,3*np.pi,abs(curve).max()+1e-3]), maxfev=6000)
    except Exception:
        return None, None
    fit=dvm(theta,*popt); ss=np.sum((curve-np.mean(curve))**2)
    r2=1-np.sum((curve-fit)**2)/ss if ss>0 else 0
    a1,a2,kappa,tp,base=popt
    hwhm=np.degrees(np.arccos(np.clip(1+np.log(0.5)/kappa,-1,1))) if kappa>0 else np.nan
    return dict(r2=r2, hwhm=hwhm, pref_dir=np.degrees(tp)%360, amp=max(a1,a2)), popt

def vec_osi_dsi(curve, theta):
    r=np.clip(curve,0,None)
    if r.sum()==0: return 0,0,np.nan,np.nan
    return (np.abs(np.sum(r*np.exp(2j*theta)))/r.sum(),               # OSI (derived, folded)
            np.abs(np.sum(r*np.exp(1j*theta)))/r.sum(),               # DSI (native)
            np.degrees(np.angle(np.sum(r*np.exp(2j*theta))))/2 % 180,
            np.degrees(np.angle(np.sum(r*np.exp(1j*theta)))) % 360)
print("helpers ready")

## 1 · Neuropixels ecephys

Spike-rate direction tuning from the **drifting** gratings of `Control block 1`
(we select `TemporalFrequency > 0` to be explicit about using drifting, not static,
trials). Visual units assigned to a CCF area via the corrected per-probe mapping.

In [ ]:
#@title Ecephys — session sub-830851
import re
ECE = "9b9e8abe-7b43-47f1-b8e1-4114f87898a1"
fh = open_nwb("001637", ECE)
g = fh["intervals"]["Control block 1_presentations"]
O=g["Orientation"][:].astype(float); C=g["contrast"][:].astype(float); TF=g["TemporalFrequency"][:].astype(float)
t0=g["start_time"][:]
grat=(C>0)&(TF>0)                              # DRIFTING only
oris=np.unique(O[grat]); dur=float(np.median(g["stop_time"][:]-g["start_time"][:]))
tg=t0[grat]; Og=O[grat]
U=fh["units"]; st=U["spike_times"][:]; sti=U["spike_times_index"][:]
def spikes(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
n=len(sti); qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
E=fh["general/extracellular_ephys/electrodes"]; eloc=col(E,"location"); egroup=col(E,"group_name")
dev=col(U,"device_name"); eci=U["extremum_channel_index"][:]
groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
off={gn:np.where(egroup==gn)[0][0] for gn in groups}; bl={gn:int((egroup==gn).sum()) for gn in groups}
def g4(d):
    for gn in groups:
        if d[-1].lower() in gn.lower(): return gn
    return groups[0]
d2g={d:g4(d) for d in set(dev)}
u_region=np.array([eloc[off[d2g[dev[i]]]+min(int(eci[i]),bl[d2g[dev[i]]]-1)] for i in range(n)])
sel=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_region]))[0]
RESP,BASE=(0.05,dur),(-0.25,-0.02)
tc_e=np.zeros((len(sel),len(oris))); popts_e={}; rows=[]
for ui,u in enumerate(sel):
    sp=spikes(u)
    ev=((np.searchsorted(sp,tg+RESP[1])-np.searchsorted(sp,tg+RESP[0]))/(RESP[1]-RESP[0])
        -(np.searchsorted(sp,tg+BASE[1])-np.searchsorted(sp,tg+BASE[0]))/(BASE[1]-BASE[0]))
    tc_e[ui]=[ev[Og==o].mean() for o in oris]
    osi,dsi,po,pd_=vec_osi_dsi(tc_e[ui],oris); f,popt=fit_dvm(tc_e[ui],oris); popts_e[u]=popt
    r=dict(unit=u,region=u_region[u],osi=osi,dsi=dsi,pref_ori=po,pref_dir=pd_,peak=tc_e[ui].max())
    if f: r.update({f"fit_{k}":v for k,v in f.items()})
    rows.append(r)
Te=pd.DataFrame(rows)
print(f"{len(Te)} visual units | drifting {grat.sum()}tr/{len(oris)}dir | median DSI={Te.dsi.median():.2f}, "
      f"OSI(derived)={Te.osi.median():.2f}, HWHM(well-fit)={Te[Te.fit_r2>0.6].fit_hwhm.median():.0f}deg")
fh.close()

## 2 · Mesoscope 2-photon (somatic ΔF/F)

In [ ]:
#@title Mesoscope — session sub-832700, plane VISl_4
MESO = "83e0c8f3-5208-417c-87c4-bc4617b0f834"
fhm = open_nwb("001768", MESO)
gm = fhm["intervals"]["Control block 1_presentations"]
Om=gm["Orientation"][:].astype(float); Cm=gm["contrast"][:].astype(float); TFm=gm["TemporalFrequency"][:].astype(float)
t0m=gm["start_time"][:]
gr=(Cm>0)&(TFm>0); oris_m=np.unique(Om[gr]); tgm=t0m[gr]; Ogm=Om[gr]
PLANE="VISl_4"; p0=fhm["processing"][PLANE]
dff=p0["dff_timeseries"]["dff_timeseries"]; data=dff["data"][:]; tsv=dff["timestamps"][:]
soma=np.where(p0["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool))[0]
RESP_M,BASE_M=(0.1,0.9),(-0.3,0.0)
tc_m=np.zeros((len(soma),len(oris_m))); popts_m={}; rows=[]
for ui,roi in enumerate(soma):
    ev=resp_trace(data[:,roi],tsv,tgm,RESP_M,BASE_M); tc_m[ui]=[np.nanmean(ev[Ogm==o]) for o in oris_m]
    osi,dsi,po,pd_=vec_osi_dsi(tc_m[ui],oris_m); f,popt=fit_dvm(tc_m[ui],oris_m); popts_m[roi]=popt
    r=dict(roi=roi,osi=osi,dsi=dsi,pref_ori=po,pref_dir=pd_,peak=np.nanmax(tc_m[ui]))
    if f: r.update({f"fit_{k}":v for k,v in f.items()})
    rows.append(r)
Tm=pd.DataFrame(rows)
print(f"{len(Tm)} soma ROIs | median DSI={Tm.dsi.median():.2f}, OSI(derived)={Tm.osi.median():.2f}, "
      f"HWHM(well-fit)={Tm[Tm.fit_r2>0.6].fit_hwhm.median():.0f}deg")
fhm.close()

## 3 · SLAP2 (dendritic glutamate ΔF/F)

The 14 directions live in the full-field (diameter ≥ 30°) drifting gratings of the
monolithic `gratings` stream. Best-yield session (sub-796630, 2025-10-01, DMD1)
with the DMD1 timebase rebuild (see the RF notebook).

In [ ]:
#@title SLAP2 — session sub-796630 2025-10-01, DMD1
SLAP = "44871646-ca8d-440d-b970-5756ed7cb47e"
fhs = open_nwb("001424", SLAP)
gs = fhs["intervals/gratings"]
dia=gs["diameter"][:]; ori=gs["orientation"][:].astype(float); con=gs["contrast"][:].astype(float)
tf=gs["temporal_frequency"][:].astype(float); t0s=gs["start_time"][:]
big=(dia>=30)&(con>0)&(tf>0); oris_s=np.unique(np.round(ori[big],3)); Ogs=np.round(ori[big],3)
DMD="DMD1"; OFFSET=0.115
dff1=fhs[f"processing/ophys/Fluorescence_{DMD}/{DMD}_dFF"]; d=dff1["data"][:]
ts=dff1["timestamps"][:]; ts_o=fhs["processing/ophys/Fluorescence_DMD2/DMD2_dFF"]["timestamps"][:]
if ts[-1]<0.6*ts_o[-1]: ts=np.linspace(ts_o[0],ts_o[-1],d.shape[0])   # rebuild compressed timebase
tgs=t0s[big]+OFFSET
RESP_S,BASE_S=(0.05,0.35),(-0.25,-0.02)
tc_s=np.zeros((d.shape[1],len(oris_s))); popts_s={}; rows=[]
for ui in range(d.shape[1]):
    ev=resp_trace(d[:,ui],ts,tgs,RESP_S,BASE_S); tc_s[ui]=[np.nanmean(ev[Ogs==o]) for o in oris_s]
    osi,dsi,po,pd_=vec_osi_dsi(tc_s[ui],oris_s); f,popt=fit_dvm(tc_s[ui],oris_s); popts_s[ui]=popt
    r=dict(roi=ui,osi=osi,dsi=dsi,pref_ori=po,pref_dir=pd_,peak=np.nanmax(tc_s[ui]))
    if f: r.update({f"fit_{k}":v for k,v in f.items()})
    rows.append(r)
Ts=pd.DataFrame(rows)
print(f"{len(Ts)} ROIs | median DSI={Ts.dsi.median():.2f}, OSI(derived)={Ts.osi.median():.2f}, "
      f"HWHM(well-fit)={Ts[Ts.fit_r2>0.6].fit_hwhm.median():.0f}deg")
fhs.close()

## 4 · The figure — direction tuning across all three scales

Top three rows: examples selected by **von Mises fit quality** (coloured data
points + line; thin grey von Mises fit as a guide). Bottom row: population DSI,
derived OSI, and fitted tuning half-width.

In [ ]:
#@title Plot
import matplotlib.pyplot as plt, matplotlib as mpl
th_fine=np.linspace(0,2*np.pi,200)
specs=[("Neuropixels\nspikes",Te,tc_e,popts_e,"unit",oris,"#08519c",3),
       ("Mesoscope 2p\nΔF/F (soma)",Tm,tc_m,popts_m,"roi",oris_m,"#238b45",0.1),
       ("SLAP2\nglutamate ΔF/F",Ts,tc_s,popts_s,"roi",oris_s,"#d94801",0.1)]
fig=plt.figure(figsize=(13,9.2))
for ri,(name,Tdf,tcm,popts,idc,ors,color,ampmin) in enumerate(specs):
    good=Tdf[(Tdf.fit_r2>0.6)&(Tdf.fit_amp>ampmin)].sort_values("fit_r2",ascending=False).head(3)
    for j,(_,row) in enumerate(good.iterrows()):
        ax=fig.add_subplot(4,6,ri*6+j*2+1,projection="polar")
        key=row[idc]; ui=int(np.where(Tdf[idc].values==key)[0][0])
        c=np.clip(tcm[ui],0,None); thc=np.append(ors,ors[0]); cc=np.append(c,c[0])
        ax.plot(thc,cc,"o-",color=color,lw=1.8,ms=3.5)                       # data leads
        if popts.get(key) is not None:
            ax.plot(th_fine,np.clip(dvm(th_fine,*popts[key]),0,None),"-",color="0.45",lw=0.8,alpha=0.7,zorder=1)  # muted fit
        ax.set_theta_zero_location("E"); ax.set_xticks(np.linspace(0,2*np.pi,8,endpoint=False))
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.tick_params(labelsize=5)
        ax.set_title(f"DSI={row.dsi:.2f} · HWHM={row.fit_hwhm:.0f}° · R²={row.fit_r2:.2f}",fontsize=6,pad=4)
    fig.text(0.012,0.85-ri*0.205,name,rotation=90,va="center",ha="center",fontsize=9,fontweight="bold",color=color)
axA=fig.add_subplot(4,3,10); axB=fig.add_subplot(4,3,11); axC=fig.add_subplot(4,3,12)
for name,Tdf,_,_,_,_,color,_ in specs:
    lab=name.split("\n")[0]
    axA.hist(Tdf.dsi.dropna(),bins=np.linspace(0,1,20),histtype="step",lw=2,color=color,density=True,label=f"{lab} ({Tdf.dsi.median():.2f})")
    axB.hist(Tdf.osi.dropna(),bins=np.linspace(0,1,20),histtype="step",lw=2,color=color,density=True,label=f"{lab} ({Tdf.osi.median():.2f})")
    w=Tdf[Tdf.fit_r2>0.6]
    axC.hist(w.fit_hwhm.dropna(),bins=np.arange(0,91,7.5),histtype="step",lw=2,color=color,density=True,label=f"{lab} ({w.fit_hwhm.median():.0f}°)")
axA.set_xlabel("DSI (direction selectivity)"); axA.set_ylabel("density"); axA.set_title("A · Direction selectivity",fontsize=9,loc="left"); axA.legend(fontsize=6)
axB.set_xlabel("OSI (derived, folded 360→180°)"); axB.set_title("B · Orientation selectivity (derived)",fontsize=9,loc="left"); axB.legend(fontsize=6)
axC.set_xlabel("tuning half-width HWHM (°)"); axC.set_title("C · Tuning width (von Mises fit)",fontsize=9,loc="left"); axC.legend(fontsize=6)
fig.suptitle("Direction tuning across three recording scales — drifting gratings, 14 directions",fontsize=11,y=0.985)
fig.subplots_adjust(left=0.06,right=0.98,top=0.93,bottom=0.065,wspace=0.42,hspace=0.72)
fig.savefig("direction_tuning_three_modalities.png",dpi=200,bbox_inches="tight")
plt.show()

## Takeaway

All three modalities show **strong, well-formed direction tuning** with realistic
half-widths (16–28° HWHM) — not razor-thin artifacts of the selection.

| Modality | median DSI | OSI (derived) | tuning HWHM | % direction-selective (DSI>0.5) |
|---|---|---|---|---|
| Neuropixels (spikes) | 0.19 | 0.39 | 28° | 11 % |
| Mesoscope (ΔF/F soma) | 0.42 | 0.64 | 16° | 53 % |
| SLAP2 (glutamate) | 0.31 | 0.49 | 21° | 24 % |

Most cells are **orientation-selective** (two roughly equal lobes → high derived
OSI, low DSI); a substantial **direction-selective** minority (one dominant lobe)
is largest in mesoscope. Two method choices keep this honest and mirror the RF
notebook: (1) we distinguish **direction** (measured, drifting gratings) from
**orientation** (derived by folding, since no static sweep exists), and (2) we
select examples by **fit quality** and report **tuning width**, so genuinely narrow
curves are shown as narrow rather than driving the selection. Together with the RF
check (spatial sensitivity), this validates that the pipeline reads real visual
feature-tuning at all three scales.